## Subnetting — reading and counting `172.30.240.133/20`

**Support context:** firewall rules, routes and `ip addr` all say things like `172.30.240.133/20`. To use them you must answer four questions fast: *which network is this host in? what is the first and last address? how many hosts fit? is another IP inside the same range?* This notebook gives you a fixed method for that. It builds on `IP-Subnet-Mask.ipynb` and `IP-Subnet-Mask-CIDR.ipynb`.

### 1. Two corrections to make first

**a) `172.30.240.133/20` is a Host** in a network. The **Network Address** is `172.30.240.0`, where **all the host bits are zero**. 

`172.30.240.133` is not the Network Address. In `172.30.240.133/20` the `.133` part marks a **host**, a machine inside the network. The **network address** (network ID) is the same address with **all host bits** set to `0`(`00000000`): `172.30.240.0`.

> The notation `address/N` is "a **host** and the **size of the network** it lives in" and `ip addr` shows it that way.

**b) How the number of subnets and their size change.** Starting point: `/16` is **one** network with 65,536 addresses (hosts).
> Each extra bit you add to the mask:
- **doubles** the number of subnets, and
- **halves** the size of each.

> **Subnetting** is done by **changing** the defatult subnet mask by **borrowing** some of the **bits** that were designed for host portion & using them to create **subnets**.

So going from `/16` to `/20` borrows 4 bits: $2⁴ = 16$ subnets of $65,536 ÷ 16 = 4,096$ addresses each. The table below (using `172.30.0.0/16`) lets you check:

|Mask|Bits borrowed|Subnets|Addresses per subnet|Usable hosts (−2)|
|---|---|---|---|---|
|`/16`|0|1|65,536|65,534|
|`/17`|1|2|32,768|32,766|
|`/18`|2|4|16,384|16,382|
|`/19`|3|8|8,192|8,190|
|`/20`|4|**16**|**4,096**|4,094|
|`/21`|5|32|2,048|2,046|
|`/22`|6|64|1,024|1,022|
|`/23`|7|128|512|510|
|`/24`|8|256|256|254|

> **Why −2:** in every subnet the first address (all host bits `0`) is the **network address** and the last (all host bits `1`) is the **broadcast**. Neither can be given to a host. Your notes say "0s and 1s are reserved for the broadcast and network address *respectively*", but it is the other way round: zeros are the network address, ones are the broadcast.

### 2. What is subnetting? Is every network a subnet?

**Subnetting** = cutting one block of addresses into smaller blocks by moving the dividing line between the **network part** and the **host part** to the **right** (borrowing host bits).

**"Subnet" is a relative word.** A network is a subnet *of* any larger block that fully contains it. Your WSL network sits inside bigger ones:

```shell
172.30.240.0/20     ← the network eth0 lives in (4,096 addresses)
 └ inside 172.30.0.0/16    (65,536)
    └ inside 172.16.0.0/12 (172.16.0.0 – 172.31.255.255, the private "Class B" block)
```
and from your ticket:
```shell
10.20.2.0/24 ⊂ 10.20.0.0/16 ⊂ 10.0.0.0/8
```

> Two rules make this easy to reason about:
1. **Blocks never partially overlap.** Two CIDR blocks are either nested (one inside the other) or completely separate.
2. **A smaller `/N` value is a bigger block.** `/16` contains `/20` contains `/24`.

**Why it matters in support:** two hosts in the **same subnet** talk directly. Hosts in **different subnets** go through the **default gateway** (router). In your ticket `app01` is in `10.20.1.0/24` and `db01` is `10.20.0.15`, which is a different subnet, so traffic goes via `10.20.1.1`. That is the single router hop you saw in `traceroute`, and the reason `ttl=63`.

### 3. Mask vs CIDR: two spellings of the same thing
#### 8 Bit Octet Chart

|128|64|32|16|8|4|2|1|
|---|---|---|---|---|---|---|---|

**The CIDR number `N` is the number of leading `1` bits in the mask.** Eight possible values for the "interesting" octet:

|Octet value in mask|Binary|`1` bits in this octet|Block size (256 − value)|
|---|---|---|---|
|128|`10000000`|1|128|
|192|`11000000`|2|64|
|224|`11100000`|3|32|
|240|`11110000`|4|16|
|248|`11111000`|5|8|
|252|`11111100`|6|4|
|254|`11111110`|7|2|
|255|`11111111`|8|1|

Examples:
- `255.255.255.0` = 8 + 8 + 8 + 0 = **/24**
- `255.255.240.0` = 8 + 8 + 4 + 0 = **/20**. This is the `netmask 255.255.240.0` that `ifconfig` showed in `IP-Address.ipynb` for the same `eth0`.
- `255.255.224.0` = 8 + 8 + 3 + 0 = **/19**, the "tricky" one in your mask notes. Host bits = 32 − 19 = 13.

> **Block size** is the number that does all the work: the **size of one subnet** *measured in the interesting octet*.

Example, mask `255.255.255.224`
- 8 + 8 + 8 + 3 = **/27** CIDR
- **256 - 224 = 32 - Block size**
- Host bits are 32 - 27 = 5, so compute the number of addresses: 2⁵ = 32 addresses with **30 usable hosts** (Counting from the full 32 bits)

### 4. The method: read any `IP/N`

Find the **interesting octet**: the one where the fixed bits end. It is octet number `ceil(N / 8)`.

|N range|Interesting octet|
|---|---|
|1–8|1st|
|9–16|2nd|
|17–24|3rd|
|25–32|4th|

Take **`172.30.240.133/20`**:

1. **Interesting octet:** 20 is between 17 and 24, so it is the **3rd** (value `240`). Octets before it are fixed. Octets after it are free.
2. **Bits in that octet:** 20 − 16 = **4** (the first two octets are fully fixed).
3. **Block size:** 2^(8−4) = **16**, which is also 256 − 240.
4. **Network start:** the largest multiple of 16 that is ≤ 240. That is 240 itself (15 × 16). Network address: **`172.30.240.0`**.
5. **Next network starts at** 240 + 16 = 256, which doesn't exist, so this one runs to the end: last address (broadcast) = **`172.30.255.255`**. This matches the `brd 172.30.255.255` in your `ip addr`.
6. **Usable range:** `172.30.240.1` to `172.30.255.254`. Host bits = 32 − 20 = 12, so 2¹² = 4,096 addresses and **4,094 usable**.

Check against your own network: the Windows host adapter `172.30.240.1` (mask `255.255.240.0`) is in this same `/20`. It is the default gateway of your WSL, which is why `ip route` shows `default via 172.30.240.1`.

**Example 2: `192.168.10.77/26`**
- Interesting octet: 4th. Bits in it: 26 − 24 = 2. Block size: 2^(8−2) = **64**.
- Multiples of 64: 0, 64, 128, 192. The largest ≤ 77 is **64**.
- Network `192.168.10.64`, next starts at 128, so broadcast `192.168.10.127`. Usable `.65 – .126` = **62 hosts**.

**Example 3: `172.16.37.9/19`**
- Interesting octet: 3rd. Bits: 19 − 16 = 3. Block size: 2^(8−3) = **32**.
- Largest multiple of 32 ≤ 37 is **32**. Network `172.16.32.0`, next at 64, so broadcast `172.16.63.255`.
- Usable `172.16.32.1 – 172.16.63.254`, 2¹³ − 2 = **8,190 hosts**.

> **Same subnet test:** two IPs are in the same subnet if they give the **same network address** under the same mask.

### Two powers of two cover everything.

Remember the table: **2¹=2, 2²=4, 2³=8, 2⁴=16, 2⁵=32, 2⁶=64, 2⁷=128, 2⁸=256**.

- **Subnets** = 2^(borrowed bits). Borrowed bits = new prefix − old prefix.
- **Addresses per subnet** = 2^(host bits). Host bits = 32 − new prefix. Usable = that number − 2.

#### Example 1: given `/27`, count forward

Starting from `10.50.0.0/24`:

- Borrowed bits: 27 − 24 = **3**, so subnets = 2³ = **8**.
- Host bits: 32 − 27 = **5**, so addresses = 2⁵ = **32**, and usable = 32 − 2 = **30**.
- Check: 8 subnets × 32 addresses = **256**, which is the whole `/24`. If the product doesn't equal the parent, one of the two numbers is wrong.

#### Example 2: given the requirement, find the prefix (your task)

"I need 5 subnets."

1. Find the smallest power of two that is ≥ 5: 2² = 4 is too small, 2³ = 8 works. So **3 borrowed bits**.
2. New prefix = 24 + 3 = **/27**.
3. Size follows: host bits = 32 − 27 = 5, so 2⁵ = **32 addresses** per subnet (30 usable).

> Counting from the full 32 bits.

Another version of the same problem: "I need 50 hosts per subnet."

1. Find host bits `h` with 2^h − 2 ≥ 50: 2⁵ − 2 = 30 is too small, 2⁶ − 2 = 62 works. So **h = 6**.
2. Prefix = 32 − 6 = **/26**, which gives 64 addresses per subnet.
3. Subnets from a `/24` = 2^(26−24) = 2² = **4**.

**Direction summary:** subnet count and host count are in tension. Each bit you borrow doubles the first and halves the second, and the product stays at 256 for a `/24`.

### 5. How to subnet: split a block

**Method:** decide how many subnets or hosts you need, then choose the mask.

- **Need N subnets?** Borrow `b` bits where 2^b ≥ N. For 3 subnets, b = 2 (gives 4).
- **Need H hosts per subnet?** Keep `h` host bits where 2^h − 2 ≥ H. For 50 hosts, h = 6 (62 usable), so the mask is 32 − 6 = **/26**.

**Example: split `192.168.10.0/24` into 4 subnets.** Borrow 2 bits, so `/26` (block size 64):

|Subnet|Network|Usable range|Broadcast|
|---|---|---|---|
|1|`192.168.10.0/26`|`.1 – .62`|`.63`|
|2|`192.168.10.64/26`|`.65 – .126`|`.127`|
|3|`192.168.10.128/26`|`.129 – .190`|`.191`|
|4|`192.168.10.192/26`|`.193 – .254`|`.255`|

**Example: split `172.16.0.0/16` into 4 subnets.** Borrow 2 bits, so `/18`, block size 64 in the **3rd** octet: `172.16.0.0/18`, `172.16.64.0/18`, `172.16.128.0/18`, `172.16.192.0/18`.

Note that the jump (the block size) is applied to whichever octet is interesting.

### 6. Verify with Python

`ipaddress` does all of the above. Use it to check your manual work.

In [1]:
import ipaddress as i

# Read a host/prefix: the network it belongs to
h = i.ip_interface('172.30.240.133/20')
n = h.network
print(n)                       # network address + prefix
print(n.netmask)               # 255.255.240.0
print(n[0], n[-1])             # first (network) and last (broadcast)
print(n[1], n[-2])             # first and last usable
print(n.num_addresses, n.num_addresses - 2)

# Containment: is a block inside a bigger one?
print(n.subnet_of(i.ip_network('172.30.0.0/16')))       # True
print(i.ip_address('172.30.250.9') in n)                # True

# Split a block into smaller ones
for s in i.ip_network('192.168.10.0/24').subnets(new_prefix=26):
    print(s, s[1], s[-2], s[-1])

# Go the other way
print(n.supernet(prefixlen_diff=4))                     # 172.30.0.0/16

172.30.240.0/20
255.255.240.0
172.30.240.0 172.30.255.255
172.30.240.1 172.30.255.254
4096 4094
True
True
192.168.10.0/26 192.168.10.1 192.168.10.62 192.168.10.63
192.168.10.64/26 192.168.10.65 192.168.10.126 192.168.10.127
192.168.10.128/26 192.168.10.129 192.168.10.190 192.168.10.191
192.168.10.192/26 192.168.10.193 192.168.10.254 192.168.10.255
172.30.0.0/16


### 7. Support notes

- **Gotcha: host bits set.** `ip_network('10.0.0.5/24')` is an error in Python (and an invalid rule in most firewalls), because `.5` is a host. It is fine as `ip_interface`.
- **`/32`** is a single host (used in `ip addr` for the `10.255.255.254/32` resolver address on `lo`). **`/0`** is everything (`0.0.0.0/0` = default route).
- **`/31`** has two addresses and is used for point-to-point links, so the "−2" rule doesn't apply.
- **Overlapping subnets** break routing, for example a VPN on `10.20.0.0/16` and a home network on `10.20.1.0/24`. Two blocks overlap if one contains the other.
- **A rule that's too wide or too narrow** causes the two common faults: security exposure, or an unexpected block like `app01`'s.
- Quick tool: `sudo apt install ipcalc`, then `ipcalc 172.30.240.133/20` does this arithmetic for you. Learn the method first. Python is already on your machine.

### TRY YOURSELF

Work each answer out by hand with the method in section 4. Then check with the Python snippet.

1. For `10.20.1.15/24`: what is the network address, broadcast, and number of usable hosts?
2. For `172.16.37.200/19`: what are the block size, network address and broadcast?
3. Convert `255.255.255.224` to CIDR, and say how many usable hosts it gives.
4. Are `192.168.10.60` and `192.168.10.70` in the same subnet with mask `/26`? And with `/25`?
5. You need 5 subnets from `10.50.0.0/24`. Which prefix do you pick, how big is each, and what are the first three subnet network addresses?
6. Your firewall allows `172.30.240.0/20`. Is `172.30.250.9` allowed? Is `172.30.2.1`?
7. A ticket says "the VPN user at `10.20.1.50` can't reach the server at `10.20.1.200`, mask `255.255.255.128`". Are they in the same subnet? What would that mean?

*(write your answers and Python checks below)*

**Task 1**:

For `10.20.1.15/24`: what is the network address, broadcast, and number of usable hosts?

1. The interesting octet is the **third** one. It's where the fixed bits end, at 24. This means the first 3 octets a fully fixed and no partially fixed ones. So, there is one network with 256 total addresses.
2. With mask `/24` or `255.255.255.0`:
    - network address (ID) is `10.20.1.0`
    - broadcast `10.20.1.255`
    - 254 usable hosts, from `10.20.1.1` to `10.20.1.254`

---

**Task 2**:

For `172.16.37.200/19`: what are the block size, network address and broadcast?

1. Interesting octet: 19 is between 17 & 24, so it is the 3rd (224 octet value in mask - 128 + 64 + 32 = 224; 256 - 224 = 32 block size)

`11111111.11111111.11100000.00000000` - 255.255.224.0

2. Bits in that octet: 19 - 16 = 3 (the first two octets are fully fixed)
3. Block size: 2^(8-3) = 32, which is also 256 - 224 = 32.
4. Network address: `172.16.32.0` - the very **first address** of the subnet, is the network ID since this subnet range is `32-63` and 37 is within it.
5. Broadcast: `172.16.63.255` - the **last address** of the subnet.

**Subntes**:

There is `/19` mask, meaning the block size is 32, so there are 8 subnets with ranges: 1) `0-31`, 2) `32-63`, 3) `64-95`, 4) `96-127`, 5) `128-159`, 6) `160-191`,7) `192-223`, 8) `224-255`.

**Which subnet does the address belong to?** The 8 subnets only exist when you split a bigger block into `/19` pieces. A host address like `172.16.37.200/19` lives in **exactly one** of them. **"The network address"** means the **start of the subnet that contains your address**, not any of the other seven.

Here `37` falls in `32–63`, which is your 2nd subnet. So:

- network address: `172.16.32.0`
- broadcast: `172.16.63.255`

Python gave the same answer, so you read it correctly.

**THE INTERESTING OCTET is the whole method.** The recipe:

1. **Find the interesting octet** and take its value. For `/19` that's the 3rd octet, value **37**. The octets _before_ it (`172.16`) are copied unchanged.
2. **Find the block it falls in.** Block size is 32, so the blocks are 0, 32, 64... The largest multiple of 32 that is ≤ 37 is **32**. A shortcut: `37 − (37 mod 32) = 37 − 5 = 32`, or `37 // 32 * 32`.
3. **Network address:** copy the octets before it, put that block start in the interesting octet, and set every octet **after** it to `0`. That gives `172.16.32.0`.
4. **Broadcast:** the block end in the interesting octet is `32 + 32 − 1 = 63`, and every octet after it is `255`. That gives `172.16.63.255`.

**What about the 4th octet (`200`)?** It's entirely in the host part, so it doesn't matter for the network or broadcast. It only matters for deciding _whether the address is inside the range_, and `.200` is, as are all values 0–255.

**One more detail:** `172.16.32.0` and `172.16.63.255` are the unusable "first" and "last" addresses of that subnet. The usable hosts are `172.16.32.1` to `172.16.63.254`, which is the 8,190 count in the notebook.

---

**Task 3**

Convert `255.255.255.224` to CIDR, and say how many usable hosts it gives.

128 + 64 = 192 + 32 = 224 - 3 bits in the **4th** octet means `/27` CIDR. Block size is 32 for `/27`, this gives 30 usable hosts.

In `255.255.255.224` the `224` is the **4th** octet. The first three octets are all `255` (8 bits each), and 224 adds 3 more: 8 + 8 + 8 + 3 = **/27**. Block size is 256 − 224 = 32, host bits are 32 − 27 = 5, so 2⁵ = 32 addresses and **30 usable hosts**.

💡 Why: count the full `255` octets first, then add the bits from the partial octet. Which octet the partial one is tells you the range of N. 

🔁 Support rule: find the octet that isn't `255` or `0`. That's the interesting one.

Your reasoning (128 + 64 + 32 = 224, so 3 bits) is correct.

Compare the two so the position sticks:

|Mask|Interesting octet|CIDR|Block size|Usable hosts|
|---|---|---|---|---|
|`255.255.224.0`|3rd|`/19`|32 (in 3rd octet)|8,190|
|`255.255.255.224`|4th|`/27`|32 (in 4th octet)|30|

Both have a block size of 32, but the subnets are very different sizes.

You can check it with `python3 -c "import ipaddress as i; print(i.ip_network('0.0.0.0/255.255.255.224').prefixlen)"`.
```bash
(.venv) dennie@X390:~/linux-tut$ python3 -c "import ipaddress as i; print(i.ip_network('0.0.0.0/255.255.255.224').prefixlen)"
27
```

---

In [1]:
import ipaddress as i

print(i.ip_network('0.0.0.0/255.255.255.224').prefixlen)

27


**Task 4**

Are `192.168.10.60` and `192.168.10.70` in the same subnet with mask `/26`? And with `/25`?

Part 1

`192.168.10.60` and `192.168.10.70`, `/26` mask.
- `/26` = `255.255.255.192`
- Bits in that octet: 26 - 24 = 2
- Block size: 256 - 192 = 64 (2^(8-2))
- Subnets: `0-63`, `64-127`, `128-191`, `192-255` (Each block is `64` wide: 128 + 64 − 1 = 191, and the next starts at 192.)

Which subnet does the addresses belong to?

- Here `60` falls in `0-63` subnet, which is the first subnet. So:
    - network address: `192.168.10.0`
    - broadcast: `192.168.10.63`
- `70` falls in `64-127` subnet, which is the second subnet:
    - network address: `192.168.10.64`
    - broadcast: `192.168.10.127`

- Total Addresses: 64 addresses and 62 usable.

Solution: `192.168.10.60` and `192.168.10.70` with `/26` are in **different subnets**.

Part 2

`192.168.10.60` and `192.168.10.70`, `/25` mask.
- `/25` = `255.255.255.128`
- Bits in that octet: 25 - 24 = 1
- Block size: 256 - 128 = 128
- Subets: `0-127`, `128-255`

Which subnet does the addresses belong to?
- `60` falls in `0-127` subnet, which is the first subnet:
    - network address: `192.168.10.0`
    - broadcast: `192.168.10.127`
- `70` falls in the **same** `0-127` subnet - the **first** subnet:
    - network address: `192.168.10.0`
    - broadcast: `192.168.10.127`

- Total Addresses: 128 per subnet and 126 usable.

Summary:

`/26`: different subnets (`.0` and `.64`).
`/25`: same subnet (`.0`).

> **Why it matters in support**: hosts in the same subnet **talk directly**. Hosts in different subnets go through the **gateway**, so a mask that is too long can make two machines that "should be neighbors" depend on a router, or on a firewall rule that doesn't cover them. This is the same kind of mismatch behind the `app01` ticket.

---

**Task 5**

You need 5 subnets from `10.50.0.0/24`. Which prefix do you pick, how big is each, and what are the first three subnet network addresses?

`/24` means 3 first octets are fixed. To subnet this default network (`10.50.0.0`) to 5 subnets, I need to borrow bits from the last forth octet.

- I pick `/27` prefix since it gives 8 subnets. Each borrowed bit doubles subnets and halves the number of hosts per each subnet.
- Block size: given `/27` prefix, the 3 borrowed bits result in 128 + 64 + 32 = 224, 256 - 224 = 32 block size. So each of the 8 subnets have 32 addresses with 30 usable hosts.
- First 3 subnets: `0-31`, `32-63`, `64-95` ranges suggest `10.50.0.0`, `10.50.0.32` and `10.50.0.64` first 3 subnet addresses.

✅ `/27` is the correct prefix: 2² = 4 subnets isn't enough for 5, so you need 3 borrowed bits.

✅ Block size 32, and the first three networks are `10.50.0.0`, `10.50.0.32` and `10.50.0.64`.

✅ 8 subnets. 2³ = 8. Subnet counts are always powers of two, so 6 can't happen. You get 3 spare subnets beyond the 5 you need, which is useful if the network grows.

> Subnets = 2^(borrowed bits). Borrowed bits = new prefix − old prefix.

✅ Each has 32 addresses and 30 usable hosts, because you lose the network address and the broadcast.

> Addresses per subnet = 2^(host bits). Host bits = 32 − new prefix. Usable = that number − 2.

✅ 64-95. The block is 32 wide: 64 + 32 − 1 = 95. The next subnet starts at 96.

Here are all 8 for reference:

|#|Network|Range|Broadcast|
|---|---|---|---|
|1|`10.50.0.0/27`|`.0–.31`|`.31`|
|2|`10.50.0.32/27`|`.32–.63`|`.63`|
|3|`10.50.0.64/27`|`.64–.95`|`.95`|
|4|`10.50.0.96/27`|`.96–.127`|`.127`|
|5|`10.50.0.128/27`|`.128–.159`|`.159`|
|6|`10.50.0.160/27`|`.160–.191`|`.191`|
|7|`10.50.0.192/27`|`.192–.223`|`.223`|
|8|`10.50.0.224/27`|`.224–.255`|`.255`|

🔁 **Support rule:** subnet counts and sizes are powers of two. List the blocks by adding the block size each time (0, 32, 64, 96...), and check each range's end against the next start.

You can confirm with `python3 -c "import ipaddress as i; [print(s) for s in i.ip_network('10.50.0.0/24').subnets(new_prefix=27)]"`.
```bash
(.venv) dennie@X390:~/linux-tut$ python3 -c "import ipaddress as i; [print(s) for s in i.ip_network('10.50.0.0/24').subnets(new_prefix=27)]"
10.50.0.0/27
10.50.0.32/27
10.50.0.64/27
10.50.0.96/27
10.50.0.128/27
10.50.0.160/27
10.50.0.192/27
10.50.0.224/27
```

In [1]:
import ipaddress as i

[print(s) for s in i.ip_network('10.50.0.0/24').subnets(new_prefix=27)]

10.50.0.0/27
10.50.0.32/27
10.50.0.64/27
10.50.0.96/27
10.50.0.128/27
10.50.0.160/27
10.50.0.192/27
10.50.0.224/27


[None, None, None, None, None, None, None, None]

**Task 6**

Your firewall allows `172.30.240.0/20`. Is `172.30.250.9` allowed? Is `172.30.2.1`?

1. `172.30.240.0/20` - borrows 4 bits - `255.255.240.0`

128 + 64 + 32 + 16 = 240

Block size: 256 - 240 = 16

Subnets: 2^4 = 16

- `172.30.0.0` - `172.30.15.255`
- `172.30.16.0` - `172.30.31.255`
- `172.30.32.0` - `172.30.47.255`
- `172.30.48.0` - `172.30.63.255`
- `172.30.64.0` - `172.30.79.255`
- `172.30.80.0` - `172.30.95.255`
- `172.30.96.0` - `172.30.111.255`
- `172.30.112.0` - `172.30.127.255`
- `172.30.128.0` - `172.30.143.255`
- `172.30.144.0` - `172.30.159.255`
- `172.30.160.0` - `172.30.175.255`
- `172.30.176.0` - `172.30.191.255`
- `172.30.192.0` - `172.30.207.255`
- `172.30.208.0` - `172.30.223.255`
- `172.30.224.0` - `172.30.239.255`
- `172.30.240.0` - `172.30.255.255`

If the firewal allows `172.30.240.0` network with mask `/20` this means host `172.30.250.9` is allowed too since it's inside this network. But `172.30.2.1` is in the different network `172.30.0.0/20`, so this host is not allowed.

✅ **`172.30.250.9` is allowed.** It is in `172.30.240.0 – 172.30.255.255`, the last of your 16 blocks.

✅ **`172.30.2.1` is not allowed.** It is in the first block, `172.30.0.0 – 172.30.15.255`, which is a different `/20`.

✅ Your 16-block list is also correct: each block is 16 wide and the end is always the next start minus 1.

---

**The shortcut don't list all 16 blocks**:
1. `172.30.250.9` → 3rd octet = **250**
2. `172.30.2.1` → 3rd octet = **2**

- `250 // 16 * 16 = 240`, so the block starts at `172.30.240.0`, which matches the rule. Allowed.
- `2 // 16 * 16 = 0`, so the block starts at `172.30.0.0`, which doesn't match. Blocked.

The `16` is the block size for `/20`. In the formula, `x // 16 * 16` means "the block start for x":

- `//` is integer division, which drops the remainder: `250 // 16 = 15`. (15 × 16 = 240, and 16 × 16 = 256 is too big.)
- `* 16` then turns the block number back into a block start: `15 × 16 = 240`.

The same steps for the second address: `2 // 16 = 0` (16 doesn't fit into 2), and `0 × 16 = 0`.

So the block start in the 3rd octet is 240 for the first address and 0 for the second, and the full network addresses are `172.30.240.0` and `172.30.0.0`. The rule's network is `172.30.240.0`, so the first matches and the second doesn't.

**It is the same method you used by hand**, just as one line. For `250`, you can see the blocks go 224, 240, and the next would be 256, so 240 is the largest start ≤ 250.

---

Your "borrows 4 bits" is correct when you count from `/16` (20 − 16).

Counting from the full 32 bits, the host part is 12 bits.

> 🔁 **Support rule:** to test one address against a rule, compute only that address's block start and compare it with the rule's network address.

**Task 7**

A ticket says "the VPN user at `10.20.1.50` can't reach the server at `10.20.1.200`, mask `255.255.255.128`". Are they in the same subnet? What would that mean?

1. Mask `255.255.255.128` is `/25` (8 + 8 + 8 + 1)
2. Block size: 256 - 128 = 128
2. Compute block starts:

- `10.20.1.50`: 50 // 128 * 128 = 0 - the network is `10.20.1.0/25` (range `.0-.127`, broadcast `.127`)
- `10.20.1.200`: 200 // 128 * 128 = 128 - the network is `10.20.1.128/25` (range `.128–.255`, broadcast `.255`)

> 💡 Convert the mask to CIDR first, then calculate the block size, and only then compute block starts.

So these two addresses are in different 2 subnets.

Mask `255.255.255.128` suggests there are 2 subnets, since there is only 1 bit is borrowed with `128` in the forth octet.

The VPN user's IP address and the server's one are on different sub-networks. If a host wants to communicate with another host on the different network, the data will be sent to the default gateway, which is the router. And the router will intelligently route the data to the destination. This means they communicate via the gateway, and a firewall on the other network could block traffic from other hosts.

Both hosts are in `10.20.1.x`, so someone looking at them would assume they are neighbours on a `/24`. The `/25` mask splits them into two networks that **can't talk directly**.
- Their traffic has to go through a router or gateway, and that path might have no route, or a firewall rule might block it.
- Where you'd look: `ip -4 addr show` on both hosts (is the mask what the design says?), `ip route get 10.20.1.200` from the VPN user's host (is there a gateway?), and `nc -zv -w 3` to the service port to see whether it's refused or timing out.
- The likely root cause is a **wrong or inconsistent mask on one host**, or a missing route or rule between the two `/25` subnets.

In [1]:
import ipaddress as i

# Read a host/prefix: the network it belongs to
h = i.ip_interface('172.16.37.200/19')
n = h.network
print(n)                       # network address + prefix
print(n.netmask)               # 255.255.240.0
print(n[0], n[-1])             # first (network) and last (broadcast)
print(n[1], n[-2])             # first and last usable
print(n.num_addresses, n.num_addresses - 2)

# Containment: is a block inside a bigger one?
print(n.subnet_of(i.ip_network('172.30.0.0/16')))       # True
print(i.ip_address('172.30.250.9') in n)                # True

# Split a block into smaller ones
for s in i.ip_network('192.168.10.0/24').subnets(new_prefix=26):
    print(s, s[1], s[-2], s[-1])

# Go the other way
print(n.supernet(prefixlen_diff=4))                     # 172.30.0.0/16

172.16.32.0/19
255.255.224.0
172.16.32.0 172.16.63.255
172.16.32.1 172.16.63.254
8192 8190
False
False
192.168.10.0/26 192.168.10.1 192.168.10.62 192.168.10.63
192.168.10.64/26 192.168.10.65 192.168.10.126 192.168.10.127
192.168.10.128/26 192.168.10.129 192.168.10.190 192.168.10.191
192.168.10.192/26 192.168.10.193 192.168.10.254 192.168.10.255
172.16.0.0/15
